# Reading a trip matrix critically

Somebody hands you a matrix. It holds 21,025 cells, one for every ordered pair of the 145 MSOAs in Tyne and Wear, and you are asked whether it can be used. You cannot read 21,025 cells. The honest version of that sentence is that nobody can, including whoever built it.

What replaces reading them is a short set of checks you can name before you open the file, and which apply to any matrix anyone hands you. This notebook applies them to a modelled matrix produced by the model you have already run at this scale, with the same doubly constrained structure and the same beta of 0.1185. Nothing here re-runs the balancing. The point is to read what a colleague handed over rather than to rebuild it, and thirteen more iterations of balancing would spend time the last two sections need more.

One claim runs through the whole exercise, so it is worth stating before any output appears. The margin check is necessary and it is not sufficient. Since a doubly constrained model is built to reproduce the row and column totals it was given, a matrix that fails that check has failed at the one thing its own structure promises, and nothing underneath is worth reading. A matrix that passes has demonstrated exactly one thing. It can reproduce its margins to the last trip and still be wrong in ways only its structure will reveal.

There is a second point, and it is the one I would most want you to leave with. Sparsity is not error, and a dense matrix is not a better one. Of the 21,025 ordered pairs in the Census commuting data for this area, 4,490 record no commuters at all and a further 7,640 record between one and five, carrying 18,744 trips between them, which is 5.04 per cent of the total. The modelled matrix has a positive number in every cell, 3,969 of them holding less than a single trip. Any statistic computed across all 21,025 cells is therefore a statistic about pairs where almost nobody travels.

Work through the sections in order, from the top of the page to the bottom.

## Where the files are

JupyterLite runs inside your browser. Nothing is installed on your machine and
you do not need administrator rights, which is why this page opens on a
locked-down work machine.

The notebook and its data arrived with the site, so there is nothing to download
and nothing to upload. Open the file browser - the panel down the left-hand
side, or the folder icon in the far-left sidebar if it is not showing - and you
will find this arrangement already in place:

```
matrix-review/
    reading-a-trip-matrix.ipynb
    data/
        zones_msoa.csv
        flows_msoa.csv
        trip_ends_msoa.csv
        cost_matrix_msoa.csv
        modelled_matrix_msoa.csv
```

Every path in the code below assumes it. The notebook sits at the top of the
folder and the data sits one level under it, so moving either one will break the
loading section.

**What happens to anything you change**

Because there is no server behind this, whatever you save goes into your
browser's own storage rather than onto a network drive. That has two
consequences worth taking seriously. Anything you want to keep should be
downloaded - right-click the file in the file browser and choose **Download**.
And if you clear your browsing data, or if your employer's IT policy clears it
for you, your saved work goes with it.

Do not edit the CSV files. If you want to try something out on them, duplicate
one first and work on the copy.

**Getting back to the original**

Should you change the notebook and want the version you started with, use
**Help > Clear Browser Data**. But read the warning it gives you before
confirming. It removes everything you have stored on this site, for every
notebook here, and it cannot be undone, so download anything you care about
first.

## Checking the files are where you think they are

Run the cell below before anything else. It reports what it can see, which is
faster than reading an error message later and guessing what went wrong.

In [ ]:
import os

DATA_FOLDER = "data"

expected = [
    "zones_msoa.csv",
    "flows_msoa.csv",
    "trip_ends_msoa.csv",
    "cost_matrix_msoa.csv",
    "modelled_matrix_msoa.csv",
]

print("Looking in:", os.path.abspath(DATA_FOLDER))
print()

if not os.path.isdir(DATA_FOLDER):
    print("That folder does not exist yet.")
    print("Check the folder names and check where this notebook is saved.")
else:
    found = sorted(os.listdir(DATA_FOLDER))
    for name in expected:
        status = "found" if name in found else "MISSING"
        print(f"  {name:28s} {status}")

## Parameters

This is the only cell in the notebook you will change. Everything below it reads
these four values and does as it is told.

Two of them move. SMALL_CELL sets what counts as a small cell in the sparsity
section, and TOP_FLOWS sets how many of the largest flows each ranked table
shows; both are there for you to push around once you have seen what they do at
their starting values. The other two stay where they are. READING_TOLERANCE is
the size of margin gap, in trips, below which a gap is rounding rather than a
fault, and one trip is generous for a file rounded to two decimal places across
145 cells. TOP_ORIGINS fixes the length of the trip-length comparison in the
fourth section, where ten zones is already more than the pattern needs.

In [ ]:
# ---------------------------------------------------------------------------
# PARAMETERS
# ---------------------------------------------------------------------------

SMALL_CELL = 5           # a cell holding this many trips or fewer counts
                         # as small, in trips

TOP_FLOWS = 10           # how many of the largest flows each table lists

READING_TOLERANCE = 1.0  # a margin gap smaller than this, in trips, is
                         # rounding in a file stored to two decimal places

TOP_ORIGINS = 10         # how many origin zones the trip-length comparison
                         # ranks

# ---------------------------------------------------------------------------

## Loading the data

Five files go in. The zone list fixes the order of everything else, so that row
three of the cost matrix and row three of the modelled matrix refer to the same
place. What gets read from each matters, because a column name is the sort of
thing that goes wrong quietly: `zone_id`, `zone_name` and `local_authority` from
the zone list, `origin_id`, `destination_id` and `trips` from the flows file,
`origin_id`, `destination_id` and `gc_min` from the cost matrix, and `zone_id`,
`resident_workers` and `jobs` from the trip ends. The `jobs` column lives in the
trip-ends file rather than in the opportunities file, which counts something
different.

The fifth file, `modelled_matrix_msoa.csv`, is the one under review. It arrives
already in the shape of a matrix, 145 rows of 145 columns with `origin_id` down
the side, which is how a colleague would send you one.

This is the slowest cell in the notebook. The cost matrix is 21,025 rows, so give
it a moment before deciding it has stalled.

In [ ]:
import numpy as np
import pandas as pd

zones = pd.read_csv(f"{DATA_FOLDER}/zones_msoa.csv", encoding="utf-8-sig")
flows = pd.read_csv(f"{DATA_FOLDER}/flows_msoa.csv", encoding="utf-8-sig")
costs = pd.read_csv(f"{DATA_FOLDER}/cost_matrix_msoa.csv", encoding="utf-8-sig")
trip_ends = pd.read_csv(f"{DATA_FOLDER}/trip_ends_msoa.csv", encoding="utf-8-sig")
supplied = pd.read_csv(f"{DATA_FOLDER}/modelled_matrix_msoa.csv",
                       encoding="utf-8-sig", index_col="origin_id")

zone_ids = list(zones["zone_id"])
zone_names = dict(zip(zones["zone_id"], zones["zone_name"]))
authority = dict(zip(zones["zone_id"], zones["local_authority"]))

print(f"Zones loaded:              {len(zone_ids)}")
print(f"Flow records:              {len(flows):,}")
print(f"Cost matrix records:       {len(costs):,}")
print(f"Trip end records:          {len(trip_ends)}")
print(f"Supplied matrix:           {supplied.shape[0]} rows by {supplied.shape[1]} columns")

In [ ]:
observed = (flows
            .pivot(index="origin_id", columns="destination_id", values="trips")
            .reindex(index=zone_ids, columns=zone_ids)
            .fillna(0)
            .values.astype(float))

cost = (costs
        .pivot(index="origin_id", columns="destination_id", values="gc_min")
        .reindex(index=zone_ids, columns=zone_ids)
        .values.astype(float))

modelled = supplied.reindex(index=zone_ids, columns=zone_ids).values.astype(float)

margins = trip_ends.set_index("zone_id").reindex(zone_ids)
origins = margins["resident_workers"].values.astype(float)
destinations = margins["jobs"].values.astype(float)

short = {z: (zone_names[z][:30]) for z in zone_ids}

print("Observed matrix:", observed.shape)
print("Modelled matrix:", modelled.shape)
print("Cost matrix:    ", cost.shape)
print()
print("Any missing values in the supplied matrix:", bool(np.isnan(modelled).any()))

## The margins, applied

State the rule before you look at the output, because a check you formulate
afterwards is not a check.

The rule is this. Every origin zone's modelled outward trips must equal its
resident workers, and every destination zone's modelled inward trips must equal
its jobs. That is not an aspiration of a doubly constrained model. It is the
definition of one, enforced by the balancing you watched converge in thirteen
iterations, and a matrix that fails it has not produced a poor forecast so much
as failed to be the kind of object it claims to be.

Two practical notes before the numbers. The file is stored to two decimal
places, so summing 145 rounded values leaves a residue of up to a tenth of a
trip in any margin; that is arithmetic, not a fault, and READING_TOLERANCE is
set at one trip to keep it out of the way. And a gap of the same size appearing
in one row and one column is worth more than two separate observations, because
a single wrong cell is the only thing that produces it.

The cell below prints the totals, counts the zones outside tolerance, and lists
the five largest gaps on each margin. Printing all 290 would fill the screen and
would not be read.

In [ ]:
row_total = modelled.sum(axis=1)
col_total = modelled.sum(axis=0)
row_gap = row_total - origins
col_gap = col_total - destinations

print("TOTALS")
print("-" * 68)
print(f"Trips in the observed matrix:      {observed.sum():>14,.0f}")
print(f"Trips in the supplied matrix:      {modelled.sum():>14,.2f}")
print(f"Resident workers, all 145 zones:   {origins.sum():>14,.0f}")
print(f"Jobs, all 145 zones:               {destinations.sum():>14,.0f}")
print()

rows_out = np.abs(row_gap) >= READING_TOLERANCE
cols_out = np.abs(col_gap) >= READING_TOLERANCE

print("MARGIN CHECK")
print("-" * 68)
print(f"Tolerance for reading a rounded file: {READING_TOLERANCE} trips")
print(f"Origin zones outside tolerance:       {int(rows_out.sum())} of 145")
print(f"Destination zones outside tolerance:  {int(cols_out.sum())} of 145")
print()

def gap_table(gap, target, label):
    order = np.argsort(-np.abs(gap))[:5]
    print(f"Five largest {label} gaps")
    print(f"  {'zone':32s} {'target':>10s} {'matrix':>12s} {'gap':>10s}")
    for i in order:
        print(f"  {short[zone_ids[i]]:32s} {target[i]:>10,.0f} "
              f"{target[i] + gap[i]:>12,.2f} {gap[i]:>10,.2f}")
    print()

gap_table(row_gap, origins, "row (outward trips against resident workers)")
gap_table(col_gap, destinations, "column (inward trips against jobs)")

In [ ]:
flagged_rows = [zone_ids[i] for i in np.where(rows_out)[0]]
flagged_cols = [zone_ids[j] for j in np.where(cols_out)[0]]

if not flagged_rows and not flagged_cols:
    print("Every margin sits inside tolerance. Nothing to look along.")
else:
    print("WHERE A MARGIN IS OUT, LOOK ALONG IT")
    print("-" * 74)
    print("The largest cells in each zone that failed, with the observed")
    print("Census figure for the same pair beside them.")
    print()

for z in flagged_rows:
    i = zone_ids.index(z)
    order = np.argsort(-modelled[i])[:5]
    print(f"Outward from {zone_names[z]} ({z})")
    print(f"  {'to':32s} {'matrix':>10s} {'observed':>10s} {'cost, min':>11s}")
    for j in order:
        print(f"  {short[zone_ids[j]]:32s} {modelled[i, j]:>10,.2f} "
              f"{observed[i, j]:>10,.0f} {cost[i, j]:>11,.2f}")
    print()

for z in flagged_cols:
    j = zone_ids.index(z)
    order = np.argsort(-modelled[:, j])[:5]
    print(f"Inward to {zone_names[z]} ({z})")
    print(f"  {'from':32s} {'matrix':>10s} {'observed':>10s} {'cost, min':>11s}")
    for i in order:
        print(f"  {short[zone_ids[i]]:32s} {modelled[i, j]:>10,.2f} "
              f"{observed[i, j]:>10,.0f} {cost[i, j]:>11,.2f}")
    print()

## Structure: the big flows and the diagonal

Two properties of a commuting matrix are worth looking at before anything else,
and both are numbers rather than impressions.

The first is where the volume sits. Commuting matrices are concentrated, and in
this one 980 cells out of 21,025 carry half of all 371,585 trips. A model that
gets the top of that distribution roughly right and the bottom badly wrong is a
different proposition from one that does the reverse, and the totals will not
tell you which you have.

The second is the diagonal. An intrazonal cell counts people who live and work
in the same zone, and its share of the total is the matrix's self-containment.
The observed figure here is 8.94 per cent. At the fifteen-zone aggregation of
exactly the same population it is 33.78 per cent, because merging Killingworth
with Shiremoor turns a commute between two zones into a commute inside one. The
number is a property of the zone system before it is a property of anybody's
travel behaviour, which is why a self-containment figure quoted without its zone
system tells you nothing.

In [ ]:
def rank_cells(matrix, label, other, other_label):
    order = np.argsort(matrix, axis=None)[::-1][:TOP_FLOWS]
    print(f"The {TOP_FLOWS} largest {label} flows")
    print(f"  {'origin':30s} {'destination':30s} {label:>10s} {other_label:>10s} {'cost':>7s}")
    for k in order:
        i, j = np.unravel_index(k, matrix.shape)
        print(f"  {short[zone_ids[i]]:30s} {short[zone_ids[j]]:30s} "
              f"{matrix[i, j]:>10,.1f} {other[i, j]:>10,.1f} {cost[i, j]:>7.2f}")
    print()

def cells_carrying_half(matrix):
    ranked = np.sort(matrix, axis=None)[::-1]
    return int(np.searchsorted(np.cumsum(ranked), matrix.sum() / 2.0)) + 1

print("CONCENTRATION")
print("-" * 66)
print(f"Cells needed to carry half the observed trips: "
      f"{cells_carrying_half(observed):,} of {observed.size:,}")
print(f"Cells needed to carry half the matrix trips:   "
      f"{cells_carrying_half(modelled):,} of {modelled.size:,}")
print()

rank_cells(observed, "observed", modelled, "matrix")
rank_cells(modelled, "matrix", observed, "observed")

In [ ]:
obs_diagonal = np.diag(observed)
mod_diagonal = np.diag(modelled)

print("SELF-CONTAINMENT")
print("-" * 70)
print(f"Observed trips staying inside their own zone: {obs_diagonal.sum():>12,.0f}"
      f"  ({obs_diagonal.sum() / observed.sum():.2%})")
print(f"Matrix trips staying inside their own zone:   {mod_diagonal.sum():>12,.2f}"
      f"  ({mod_diagonal.sum() / modelled.sum():.2%})")
print()

obs_share = obs_diagonal / observed.sum(axis=1)
mod_share = mod_diagonal / modelled.sum(axis=1)
order = np.argsort(-obs_share)[:TOP_FLOWS]

print(f"The {TOP_FLOWS} most self-contained zones, by observed share")
print(f"  {'zone':32s} {'observed':>10s} {'matrix':>10s}")
for i in order:
    print(f"  {short[zone_ids[i]]:32s} {obs_share[i]:>10.1%} {mod_share[i]:>10.1%}")

## Sparsity, and what a small number is worth

Nearly a fifth of this matrix is empty. Reading the count on its own invites the
conclusion that something has gone missing, so read it beside the volume those
cells would have carried, which is the figure that decides whether the emptiness
matters.

The Census file `flows_msoa.csv` holds 16,535 rows for a matrix of 21,025 pairs.
The 4,490 absent pairs are not errors and they were not dropped in processing:
in 2011 nobody was recorded commuting from those origins to those destinations,
and a pair with no commuters has no row. A modelled matrix behaves differently.
The gravity form multiplies three positive quantities together, so it cannot
produce a zero, and it fills the empty pairs with fractions of a person.

Change SMALL_CELL and re-run this section. What happens to the count of small
cells is obvious. What happens to the share of trips they carry is the answer
you want.

In [ ]:
print("OBSERVED MATRIX")
print("-" * 70)
zero_cells = int((observed == 0).sum())
small = (observed > 0) & (observed <= SMALL_CELL)
def line(label, value, note):
    print(f"{label:<42s} {value:>10} {note}".rstrip())

line("Cells in the matrix:", f"{observed.size:,}", "")
line("Cells recording no commuters:", f"{zero_cells:,}",
     f"({zero_cells / observed.size:.1%})")
line(f"Cells holding 1 to {SMALL_CELL} trips:", f"{int(small.sum()):,}",
     f"({small.sum() / observed.size:.1%})")
line("Trips carried by those small cells:", f"{observed[small].sum():,.0f}",
     f"({observed[small].sum() / observed.sum():.2%} of all trips)")
print()

print("SUPPLIED MATRIX, ON THE SAME BASIS")
print("-" * 70)
tiny = modelled < 1
small_m = (modelled > 0) & (modelled <= SMALL_CELL)
line("Cells holding exactly zero:", f"{int((modelled == 0).sum()):,}", "")
line("Cells holding less than one trip:", f"{int(tiny.sum()):,}",
     f"({tiny.sum() / modelled.size:.1%})")
line("Trips carried by those cells:", f"{modelled[tiny].sum():,.1f}",
     f"({modelled[tiny].sum() / modelled.sum():.2%} of all trips)")
line(f"Cells holding {SMALL_CELL} trips or fewer:", f"{int(small_m.sum()):,}",
     f"({small_m.sum() / modelled.size:.1%})")
line("Trips carried by those cells:", f"{modelled[small_m].sum():,.1f}",
     f"({modelled[small_m].sum() / modelled.sum():.2%} of all trips)")

## The matrix against the Census

Here is the comparison people reach for at work, and here is why I think it is
the weakest of the ones available. Take the modelled value and the observed
value in each of the 21,025 cells, correlate them, and square the result. The
number that comes back is printed below, and it sounds like agreement. Most of
what it measures is the matrix and the Census agreeing that almost nobody
commutes from Chopwell to Sunderland, which is true, and repeated across the four
thousand empty pairs that dominate the calculation. Restrict the
same correlation to the 1,712 cells carrying fifty trips or more, which between
them account for 63 per cent of all commuting in the area, and it falls. A
goodness-of-fit statistic that improves when you include the cells nobody
travels in is not measuring what its user thinks.

So compare at corridor level instead. The first table below aggregates all
21,025 cells into the 25 ordered pairs of local authorities, which is coarse
enough to read and fine enough to show where the model puts people who are not
there. The second takes a different cut: for each origin zone, the mean
generalised cost of a trip leaving it, in the matrix and in the Census. That
statistic is sensitive to something the corridor table hides, because a zone
whose trips have been sent to the wrong end of Tyne and Wear will show a mean
trip cost that no amount of margin balancing would disturb.

Read the second table for outliers rather than for the general pattern. A zone
sitting three or four minutes away from its observed mean is ordinary. A zone
sitting twenty-eight minutes away is not a modelling result.

In [ ]:
flat_obs = observed.ravel()
flat_mod = modelled.ravel()
r_all = np.corrcoef(flat_obs, flat_mod)[0, 1]

busy = observed >= 50
r_busy = np.corrcoef(observed[busy], modelled[busy])[0, 1]

print("CORRELATION")
print("-" * 66)
print(f"Across all {observed.size:,} cells:          r = {r_all:.4f}   r-squared = {r_all ** 2:.4f}")
print(f"Across the {int(busy.sum()):,} cells with 50+ trips: "
      f"r = {r_busy:.4f}   r-squared = {r_busy ** 2:.4f}")
print(f"Share of all trips in those cells:  {observed[busy].sum() / observed.sum():.1%}")
print()

la_index = {z: authority[z] for z in zone_ids}
la_names = sorted(set(la_index.values()))
position = {name: k for k, name in enumerate(la_names)}
rows = np.array([position[la_index[z]] for z in zone_ids])

obs_corridor = np.zeros((len(la_names), len(la_names)))
mod_corridor = np.zeros((len(la_names), len(la_names)))
for a in range(len(la_names)):
    for b in range(len(la_names)):
        block = np.ix_(rows == a, rows == b)
        obs_corridor[a, b] = observed[block].sum()
        mod_corridor[a, b] = modelled[block].sum()

corridor = pd.DataFrame({
    "observed": obs_corridor.ravel().round(0),
    "matrix": mod_corridor.ravel().round(0),
}, index=pd.MultiIndex.from_product([la_names, la_names],
                                    names=["from", "to"]))
corridor["difference"] = (corridor["matrix"] - corridor["observed"]).round(0)

print(f"The {TOP_FLOWS} local-authority corridors furthest from the Census")
print(corridor.reindex(corridor["difference"].abs()
                       .sort_values(ascending=False).index)
      .head(TOP_FLOWS)
      .to_string())

In [ ]:
mod_mean_cost = (modelled * cost).sum(axis=1) / modelled.sum(axis=1)
obs_mean_cost = (observed * cost).sum(axis=1) / observed.sum(axis=1)
cost_gap = mod_mean_cost - obs_mean_cost

print("MEAN GENERALISED COST OF A TRIP LEAVING EACH ZONE, IN MINUTES")
print("-" * 78)
print(f"All trips, matrix:   {(modelled * cost).sum() / modelled.sum():.2f}")
print(f"All trips, observed: {(observed * cost).sum() / observed.sum():.2f}")
print()

order = np.argsort(-np.abs(cost_gap))[:TOP_ORIGINS]
print(f"The {TOP_ORIGINS} origin zones furthest from their observed mean")
print(f"  {'zone':32s} {'authority':20s} {'matrix':>8s} {'observed':>9s} {'gap':>8s}")
for i in order:
    print(f"  {short[zone_ids[i]]:32s} {authority[zone_ids[i]][:20]:20s} "
          f"{mod_mean_cost[i]:>8.2f} {obs_mean_cost[i]:>9.2f} {cost_gap[i]:>8.2f}")

In [ ]:
worst = np.argsort(-np.abs(cost_gap))[:2]

print("THE TWO ZONES AT THE TOP OF THAT LIST, IN DETAIL")
print("-" * 78)
for i in worst:
    z = zone_ids[i]
    print(f"{zone_names[z]} ({z}, {authority[z]}), "
          f"{origins[i]:,.0f} resident workers")
    print()
    print(f"  Where the matrix sends them, five largest")
    print(f"    {'destination':32s} {'authority':20s} {'matrix':>9s} {'observed':>9s} {'cost':>7s}")
    for j in np.argsort(-modelled[i])[:5]:
        print(f"    {short[zone_ids[j]]:32s} {authority[zone_ids[j]][:20]:20s} "
              f"{modelled[i, j]:>9,.2f} {observed[i, j]:>9,.0f} {cost[i, j]:>7.2f}")
    print()
    print(f"  Where the Census sends them, five largest")
    print(f"    {'destination':32s} {'authority':20s} {'matrix':>9s} {'observed':>9s} {'cost':>7s}")
    for j in np.argsort(-observed[i])[:5]:
        print(f"    {short[zone_ids[j]]:32s} {authority[zone_ids[j]][:20]:20s} "
              f"{modelled[i, j]:>9,.2f} {observed[i, j]:>9,.0f} {cost[i, j]:>7.2f}")
    print()
    print(f"  Trips staying inside the zone: matrix {modelled[i, i]:,.2f}, "
          f"observed {observed[i, i]:,.0f}")
    print()

print(f"Median intrazonal cell across all 145 zones: "
      f"{np.median(np.diag(modelled)):,.2f}")

## Four questions to ask of any matrix

Everything so far has been about this file. This section is the part you will
use at work, because the four questions below are answerable about any matrix
anyone sends you, and asking them takes about a minute.

1. **What is the base year, and what was happening in it?**
2. **What is in a cell?** One trip, by whom, for what purpose, by which mode, at
   what time of day.
3. **What zone system, and on which boundaries?**
4. **What was constrained, and what was free to move?**

The answers for this dataset are recoverable from the data rather than from
anybody's memory, which is the point of the cell below. It checks each one
against the files instead of asserting it. Where a check cannot settle a
question, the cell says so and names what you would have to go and read.

Two things it will show you are worth pausing on. The generalised costs were
built from a value of time of £12.00 an hour and a vehicle operating cost of
£0.15 a kilometre, the second of which the dataset documentation records as
adding 0.75 minutes of cost per kilometre travelled.
Those money values are placeholders chosen when the dataset was assembled and
they are not from any specific release of the Department for Transport's
Transport Analysis Guidance. Treat them as an assumption to be argued with, not
as authority. The costs are also free-flow: no congestion, no junction delay, no queue at
the Tyne Tunnel and nothing that happens at half past eight on a Tuesday. The implied speeds printed
below are what free-flow looks like when you turn it back into kilometres per
hour.

In [ ]:
print("1. BASE YEAR")
print("-" * 70)
print(f"Flow records supplied:           {len(flows):,}")
print(f"Ordered pairs of 145 zones:      {len(zone_ids) ** 2:,}")
print(f"Pairs absent from the file:      {len(zone_ids) ** 2 - len(flows):,}")
print("The file itself carries no date. Census 2011 table WU03EW, taken on")
print("27 March 2011; the 2021 Census fell in the third lockdown, so its")
print("travel-to-work figures describe home working rather than commuting.")
print()

print("2. WHAT IS IN A CELL")
print("-" * 70)
print("Columns in the flows file:", ", ".join(flows.columns))
print("The file carries no mode, no purpose, no distance and no time of day.")
print("A cell is one number: usual residence to usual workplace, all modes,")
print("all day, one journey per worker. Nothing in it is a peak-hour car trip.")
print()

print("3. ZONE SYSTEM")
print("-" * 70)
print(f"Zones: {len(zone_ids)}")
print(f"Identifier pattern: {zone_ids[0]} to {zone_ids[-1]}")
print("E02 identifiers are middle layer super output areas on 2011 boundaries,")
print("which is what WU03EW was published on and what the geometry matches.")
print(f"Local authorities represented: {len(la_names)} - {', '.join(la_names)}")
print()

print("4. WHAT WAS CONSTRAINED")
print("-" * 70)
observed_rows = observed.sum(axis=1)
observed_cols = observed.sum(axis=0)
print(f"Largest gap, observed row totals against resident_workers: "
      f"{np.abs(observed_rows - origins).max():.2f}")
print(f"Largest gap, observed column totals against jobs:          "
      f"{np.abs(observed_cols - destinations).max():.2f}")
print("Both margins come from this matrix, so the model was constrained to")
print("Census 2011 trip ends. Nothing else in the matrix was constrained.")
print()

print("WHAT THE COST MATRIX ASSUMES")
print("-" * 70)
off_diagonal = ~np.eye(len(zone_ids), dtype=bool)
travel_time = (costs
               .pivot(index="origin_id", columns="destination_id", values="time_min")
               .reindex(index=zone_ids, columns=zone_ids).values.astype(float))
distance = (costs
            .pivot(index="origin_id", columns="destination_id", values="distance_km")
            .reindex(index=zone_ids, columns=zone_ids).values.astype(float))
speed = distance[off_diagonal] / (travel_time[off_diagonal] / 60.0)
print(f"Implied door-to-door speed, median: {np.median(speed):.1f} km/h")
print(f"Implied door-to-door speed, fastest pair: {speed.max():.1f} km/h")
print(f"Range across the {speed.size:,} interzonal pairs: {np.ptp(speed):.1f} km/h")
print("Free-flow, routed over OS Open Roads. No congestion is represented.")
print()
nearest = np.where(off_diagonal, cost, np.inf).min(axis=1)
print("Intrazonal generalised cost is set to half the cost of reaching the")
print(f"nearest other zone. Largest departure from that rule: "
      f"{np.abs(np.diag(cost) - 0.5 * nearest).max():.3f} minutes.")

## Writing it up

You now have what a review note needs. Two things in this matrix are wrong, they
are wrong in different ways, and only one of them appeared in the margin check.
Which check found which is the substance of the note, rather than the fact that
you found them.

Before you leave, download anything you want to keep. Nothing in this browser
persists, and a cleared cache takes it with it.